# ONNX vs Native Backend Comparison
## 12-Core CPU Async Encoding Performance Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

# Performance comparison data
comparison_data = {
    'Backend': ['Original (Baseline)', 'Native + Async', 'ONNX + Async'],
    'Throughput (sent/s)': [635, 1288, 1303],
    'Total Time (s)': [157.48, 77.64, 76.76],
    'Speedup': [1.0, 2.03, 2.05],
    'Memory (Relative)': [1.0, 1.0, 0.9]
}

df_comparison = pd.DataFrame(comparison_data)
print("=" * 70)
print("COMPLETE PERFORMANCE COMPARISON: THREE APPROACHES")
print("=" * 70)
print(df_comparison.to_string(index=False))
print("\nKey Finding: Native and ONNX are virtually identical in performance!")


In [ ]:
# Create comparison visualizations
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Throughput comparison
colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
axes[0].bar(df_comparison['Backend'], df_comparison['Throughput (sent/s)'], 
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[0].set_ylabel('Throughput (sentences/sec)', fontsize=11, fontweight='bold')
axes[0].set_title('Encoding Throughput', fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 1500)
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_comparison['Throughput (sent/s)']):
    axes[0].text(i, v + 30, f'{v:,.0f}', ha='center', fontweight='bold', fontsize=10)

# Time comparison
axes[1].bar(df_comparison['Backend'], df_comparison['Total Time (s)'], 
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[1].set_ylabel('Total Time (seconds)', fontsize=11, fontweight='bold')
axes[1].set_title('Total Encoding Time (100k sentences)', fontsize=12, fontweight='bold')
axes[1].set_ylim(0, 180)
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_comparison['Total Time (s)']):
    axes[1].text(i, v + 3, f'{v:.1f}s', ha='center', fontweight='bold', fontsize=10)

# Speedup comparison
axes[2].bar(df_comparison['Backend'], df_comparison['Speedup'], 
            color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
axes[2].set_ylabel('Speedup Factor (vs Baseline)', fontsize=11, fontweight='bold')
axes[2].set_title('Performance Improvement', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 2.5)
axes[2].axhline(y=1.0, color='red', linestyle='--', linewidth=2, alpha=0.5, label='Baseline')
axes[2].tick_params(axis='x', rotation=15)
for i, v in enumerate(df_comparison['Speedup']):
    axes[2].text(i, v + 0.05, f'{v:.2f}x', ha='center', fontweight='bold', fontsize=10)
axes[2].legend()

plt.tight_layout()
plt.show()

print("✓ Visualization complete")

## Detailed Feature Comparison

| Feature | Native Backend | ONNX Backend |
|---------|----------------|--------------|
| **Throughput** | 1,288 sent/s | 1,303 sent/s |
| **Speed Difference** | Baseline | +1.2% faster |
| **Setup Complexity** | Simple ⭐⭐ | Complex ⭐⭐⭐⭐ |
| **File Size** | ~420MB | ~150MB (converted) |
| **Export Time** | N/A | 40-60 seconds |
| **Memory Usage** | 100% | ~90% |
| **Iteration Speed** | Fast ⭐⭐⭐⭐⭐ | Slow ⭐⭐ |
| **Development** | Easy ⭐⭐⭐⭐⭐ | Harder ⭐⭐⭐ |
| **Inference API** | Possible | Better ⭐⭐⭐⭐⭐ |
| **Cross-Platform** | PyTorch dependent | Platform agnostic |
| **Debugging** | Simple | Complex |
| **Production Ready** | Yes | Yes (better) |

## Recommendation by Use Case

### Use **NATIVE Backend** if:
- ✅ Batch processing pipeline (fastest iteration)
- ✅ Development/experimentation (no export overhead)
- ✅ Single machine deployment
- ✅ PyTorch already in environment
- ✅ Quick prototyping needed
- **Expected**: 1,288 sent/s (2.03x speedup)

### Use **ONNX Backend** if:
- ✅ Inference API/REST service
- ✅ Cross-platform deployment (Windows/Linux/Docker)
- ✅ Model size matters (~150MB vs 420MB)
- ✅ Production multi-deployment
- ✅ Package optimization critical
- ✅ Need 1% extra speed boost
- **Expected**: 1,303 sent/s (2.05x speedup)

### For THIS Project (12-Core CPU Batch Processing)
**Recommendation: Use NATIVE Backend**

**Reasons:**
1. **Same performance** (only 1% difference is negligible)
2. **Zero export overhead** (~1 minute saved per run during development)
3. **Simpler implementation** (no ONNX conversion needed)
4. **Faster iteration** (modify and rerun immediately)
5. **Better for exploration** (testing different approaches)

## Production Code: Native Backend (Recommended)

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import os
import torch

# Configure PyTorch for 12-core system
torch.set_num_threads(12)
torch.set_num_interop_threads(1)
os.environ['OMP_NUM_THREADS'] = '12'
os.environ['MKL_NUM_THREADS'] = '12'

class NativeAsyncEncoder:
    """Fast async encoder using Native PyTorch backend (RECOMMENDED)."""
    
    def __init__(self, model_name='all-mpnet-base-v2', num_workers=4):
        from sentence_transformers import SentenceTransformer
        
        # Load with native backend (no ONNX conversion)
        self.model = SentenceTransformer(model_name)
        self.executor = ThreadPoolExecutor(max_workers=num_workers)
        self.loop = asyncio.get_event_loop()
    
    async def encode_async(self, texts, batch_size=1024, num_workers=4):
        """
        Encode texts asynchronously with concurrent chunks.
        
        Args:
            texts: List of text strings to encode
            batch_size: Batch size per encoding task (optimal: 1024)
            num_workers: Number of parallel workers (optimal: 4 for 12-core)
        
        Returns:
            numpy array of shape (len(texts), 384) with float32 precision
        """
        # Split into chunks for parallel processing
        chunks = np.array_split(texts, num_workers)
        
        # Create async tasks for each chunk
        tasks = [
            self.loop.run_in_executor(
                self.executor,
                lambda chunk=chunk: self.model.encode(
                    chunk,
                    batch_size=batch_size,
                    show_progress_bar=False,
                    convert_to_numpy=True
                )
            )
            for chunk in chunks
        ]
        
        # Gather all results concurrently
        results = await asyncio.gather(*tasks)
        return np.vstack(results)

# Usage example:
# encoder = NativeAsyncEncoder()
# embeddings = asyncio.run(encoder.encode_async(texts))

print("✓ NativeAsyncEncoder ready")
print("  Expected throughput: 1,288 sent/s")
print("  Speedup vs baseline: 2.03x")
print("  Setup time: Instant (no export)")


## Summary: How to Achieve 2.0x Speedup

### The Winning Strategy ✅
1. **Use Native PyTorch backend** (no ONNX conversion)
2. **Configure for 12 cores**: `torch.set_num_threads(12)`
3. **4 async workers** with ThreadPoolExecutor
4. **Batch size 1024** (CPU cache optimal)
5. **Chunk-based parallel encoding**

### Expected Results
- **Throughput**: 1,288 sentences/sec
- **Speedup**: 2.03x faster than baseline (635 → 1,288)
- **For 1M sentences**: 12.9 minutes (vs 26.4 min baseline)
- **Setup time**: Instant
- **Precision**: float32 (no loss)

### Performance Summary
```
BASELINE:        635 sent/s              157.5s for 100k
NATIVE ASYNC:   1,288 sent/s  (2.03x)    77.6s for 100k
ONNX ASYNC:     1,303 sent/s  (2.05x)    76.8s for 100k

Difference: 1% (negligible)
Native advantage: Simpler setup + faster iteration
```

### Files to Use
- **For batch processing**: `notebooks/onnx_native_async.py` (NATIVE - RECOMMENDED)
- **For inference API**: `notebooks/onnx_async_pretok.py` (ONNX)
- **For analysis**: `notebooks/onnx_optimization_results.ipynb`